# v0.2 — Modern Python foundation and clean package layout

v0.1 provisioned Azure and proved the environment worked. It left one thing unresolved: all of the reusable
code lived in a single package called `setup`, which is a name that describes when the code runs rather than
what it is responsible for. A package named after a phase of work grows by accretion, and by the end of v0.1 it
held configuration, subprocess plumbing, Azure CLI parsing, SQL validation, and timing helpers in six files.

This release replaces that with five named boundaries, adds two command line entry points, and splits the tests
into a fast offline suite and an Azure suite. No agent code is migrated here. LangGraph arrives in v0.3, and it
arrives onto a foundation that is already typed, tested, and importable.

Everything below imports the production package. Nothing of substance is defined in this notebook, which is the
property that makes a notebook trustworthy: if a cell works here, the same code works in the scripts, the
commands, and the tests.

## Setup

Run `uv sync` before opening this notebook, and select the `.venv` interpreter as the kernel.

Cells that reach Azure are written so they report a readable message instead of raising when the environment is
not provisioned, so the whole notebook runs top to bottom on a fresh clone.

In [1]:
import enterprise_agents_on_foundry
from enterprise_agents_on_foundry.config.settings import load_settings, repository_root
from enterprise_agents_on_foundry.observability.measurements import MeasurementSet

ROOT = repository_root()
settings = load_settings()
measurements = MeasurementSet(release="v0.2")

print(f"Package version : {enterprise_agents_on_foundry.__version__}")
print(f"Repository      : {ROOT}")
print(f"Environment     : {settings.environment_name}")
print(f"Provisioned     : {settings.is_provisioned}")

Package version : 0.3.0
Repository      : C:\Users\shchitt\Downloads\Projects\enterprise-agents-workspace\enterprise-agents-on-foundry
Environment     : dev
Provisioned     : True


In [6]:
import json
json_settings = json.dumps(settings.model_dump(), indent=2)
print(f"Settings:\n{json_settings}")

Settings:
{
  "project_name": "enterprise-agents-on-foundry",
  "environment_name": "dev",
  "dataset_variant": "adventureworks-lt",
  "azure_subscription_id": "1fad602f-d06f-46af-8f70-78a2c2c53b24",
  "azure_tenant_id": "16b3c013-d300-468d-ac64-7eda0820b6d3",
  "azure_location": "westus3",
  "azure_resource_group": "rg-enterprise-agents-on-foundry-dev",
  "azure_foundry_resource_name": "aif-eaof-dev-wgi4fh",
  "azure_foundry_project_name": "proj-eaof-dev",
  "azure_foundry_project_endpoint": "https://aif-eaof-dev-wgi4fh.services.ai.azure.com/api/projects/proj-eaof-dev",
  "azure_foundry_account_endpoint": "https://aif-eaof-dev-wgi4fh.cognitiveservices.azure.com/",
  "azure_model_deployment_name": "chat-model",
  "azure_model_name": "gpt-5.4-mini",
  "azure_model_version": "2026-03-17",
  "azure_model_sku_name": "GlobalStandard",
  "azure_model_capacity": 10,
  "azure_openai_api_version": "2025-04-01-preview",
  "azure_sql_server_name": "sql-eaof-dev-wgi4fh",
  "azure_sql_server_fqdn":

## 1. Why the project uses a `src/` layout

There are two ways to lay out a Python project. The flat layout puts the package directory at the repository
root. The `src/` layout puts it one level down, inside a directory that is not on `sys.path`.

The difference only shows up when something is wrong, which is exactly when it matters.

With a flat layout, running `python` from the repository root puts the root on `sys.path`, so `import
enterprise_agents_on_foundry` finds the source directory whether or not the package is installed. Tests then
pass against source files that were never packaged. A missing module in `pyproject.toml`, a file excluded by
the build backend, or a package that was never installed at all are all invisible until someone installs the
wheel somewhere else and it fails.

With a `src/` layout, the source directory is unreachable by accident. The only way to import the package is to
install it, which `uv sync` does in editable mode. What the tests import is therefore what the wheel contains.

The cell below proves the distinction: the imported module resolves through the installed distribution, and the
repository root is not what makes the import work.

In [8]:
import sys
from pathlib import Path

package_dir = Path(enterprise_agents_on_foundry.__file__).parent

print(f"Imported from : {package_dir.relative_to(ROOT)}")
print(f"src on sys.path: {str(ROOT / 'src') in sys.path}")
print()
print("The package is importable because it is installed, not because of the working directory.")
print("That is why 'uv sync' is a prerequisite rather than a convenience.")

Imported from : src\enterprise_agents_on_foundry
src on sys.path: True

The package is importable because it is installed, not because of the working directory.
That is why 'uv sync' is a prerequisite rather than a convenience.


In [9]:
# repository_root() is anchored on this file's location, not on the process working directory,
# so a notebook, a script, and a test all resolve the same paths.
import os

print(f"Notebook working directory: {Path(os.getcwd()).name}")
print(f"repository_root()         : {ROOT.name}")
print(f"pyproject.toml found      : {(ROOT / 'pyproject.toml').is_file()}")

Notebook working directory: notebooks
repository_root()         : enterprise-agents-on-foundry
pyproject.toml found      : True


## 2. What `pyproject.toml`, `uv.lock`, `.venv`, and `uv run` each do

These four are often treated as one undifferentiated blob of tooling. They answer four different questions.

`pyproject.toml` declares intent. It states which Python versions are acceptable, which dependencies are
required, which are optional, and how the build backend should package the project. It contains ranges, not
exact versions, because a library that pins exact versions cannot be composed with anything else.

`uv.lock` records a resolution. It names the exact version and hash of every package that satisfied those
ranges, for every platform. It is committed, which is what makes a build reproducible: two developers and CI
install byte-identical dependencies. The legacy project had an unpinned `requirements.txt`, so nobody could say
what was actually installed anywhere.

`.venv` is the materialisation of the lockfile on one machine. It is disposable, is not committed, and can be
deleted and rebuilt from the lockfile at any time.

`uv run` is the guarantee that a command uses that environment. It re-checks the lockfile before running, so a
stale environment is repaired rather than silently used. This is why every command in this repository is
written as `uv run ...` and never as a bare `python ...`.

In [10]:
import tomllib

pyproject = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))
project = pyproject["project"]

print(f"Name           : {project['name']} {project['version']}")
print(f"Requires Python: {project['requires-python']}")
print(f"Runtime deps   : {', '.join(project['dependencies'])}")
print(f"Optional extras: {', '.join(project.get('optional-dependencies', {}))}")
print(f"Dev group      : {', '.join(pyproject['dependency-groups']['dev'])}")
print()
print("Runtime dependencies are deliberately few. Everything needed only for development")
print("is in the dev group, so it is never installed alongside the shipped package.")

Name           : enterprise-agents-on-foundry 0.3.0
Requires Python: >=3.12
Runtime deps   : azure-identity>=1.25.3, langchain-core>=1.2.0, langchain-openai>=1.2.0, langgraph>=1.2.0, pydantic>=2.13.4, pydantic-settings>=2.14.2, python-dotenv>=1.2.2
Optional extras: database
Dev group      : ipykernel>=7.3.0, jupyter>=1.1.1, mypy>=2.3.0, nbformat>=5.10.4, pyodbc>=5.2.0, pytest>=9.1.1, ruff>=0.16.0

Runtime dependencies are deliberately few. Everything needed only for development
is in the dev group, so it is never installed alongside the shipped package.


In [11]:
lock_lines = (ROOT / "uv.lock").read_text(encoding="utf-8").splitlines()
locked_packages = sum(1 for line in lock_lines if line.strip() == "[[package]]")

print(f"Declared runtime dependencies : {len(project['dependencies'])}")
print(f"Packages pinned in uv.lock    : {locked_packages}")
print()
print("The gap between those two numbers is the transitive dependency tree.")
print("Declaring it is not the same as knowing it. The lockfile is what makes it knowable.")

Declared runtime dependencies : 7
Packages pinned in uv.lock    : 147

The gap between those two numbers is the transitive dependency tree.
Declaring it is not the same as knowing it. The lockfile is what makes it knowable.


## 3. Package modules, scripts, notebooks, and tests are four different things

A large share of the problems in the legacy backend came from putting code in the wrong one of these. The rule
this repository follows is that each has exactly one job.

A **package module** under `src/` holds logic. It is importable, typed, has no side effects at import time, and
never prints as its primary output. It returns values and raises typed errors so that a caller can decide what
to do. It is the only place where behaviour is defined.

A **script** under `scripts/` is an adapter. It parses arguments, calls package functions, prints for a human,
and returns an exit code. If a script contains a rule, that rule cannot be tested, cannot be reused by the
notebook, and will eventually be reimplemented slightly differently somewhere else. In v0.1 that had already
started: the same header-printing helper existed twice and three subprocess wrappers were near-identical.

A **notebook** teaches. It narrates decisions and shows results, and it imports the package rather than
redefining it. A notebook that defines its own version of a function is a second implementation that no test
covers and that drifts the moment the real one changes.

A **test** states a requirement. It fails when a rule is broken, and its name says which rule.

In [12]:
def _lines(path: Path) -> int:
    return sum(
        1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip() and not line.strip().startswith("#")
    )


package_files = sorted((ROOT / "src").rglob("*.py"))
script_files = sorted((ROOT / "scripts").glob("*.py"))
unit_files = sorted((ROOT / "tests" / "unit").glob("test_*.py"))
integration_files = sorted((ROOT / "tests" / "integration").glob("test_*.py"))

print(f"{'kind':<14}{'files':>6}{'lines':>8}")
print(f"{'-' * 28}")
for kind, files in [
    ("package", package_files),
    ("scripts", script_files),
    ("unit tests", unit_files),
    ("integration", integration_files),
]:
    print(f"{kind:<14}{len(files):>6}{sum(_lines(path) for path in files):>8}")

print()
print("Scripts stay small on purpose. A growing script is logic that escaped the package.")

kind           files   lines
----------------------------


package           28    2887
scripts            3     298
unit tests        10    1208
integration        2     176

Scripts stay small on purpose. A growing script is logic that escaped the package.


In [13]:
# The five boundaries, and what each one is responsible for.
for sub in sorted(path for path in (ROOT / "src" / "enterprise_agents_on_foundry").iterdir() if path.is_dir()):
    modules = sorted(path.stem for path in sub.glob("*.py") if path.stem != "__init__")
    print(f"{sub.name:<16}{', '.join(modules)}")

__pycache__     
agents          graph, model, nodes, prompts, schema_context, state
cli             console, database_info, verify
config          settings
database        connection, metadata, models, validation
infrastructure  azd_outputs, commands, environment, model_catalog
observability   measurements, timing
setup           


## 4. Typed configuration loading

Configuration has exactly one entry point: `load_settings()`. Nothing else in the package or the scripts reads
an environment variable. v0.1 was already close to this, but `scripts/preprovision_check.py` still called
`os.environ.get` four times with inline defaults, which meant the model SKU and capacity were configured in one
place and defaulted in another.

Three properties are worth being explicit about.

`Settings` is **frozen**. Configuration that can be mutated after loading is configuration that can differ
between two points in the same run, and the resulting bug is reported as intermittent.

The process environment **wins over** the `.env` file, and loading a `.env` file never mutates `os.environ`.
The legacy backend called `load_dotenv(override=True)`, which is the opposite: a stale file on a developer's
disk silently overrode what CI had deliberately exported.

Validation happens **once, at load**, and a failure names the variable without echoing its value.

In [15]:
# print(f"{'setting':<34}{'value'}")
# print("-" * 70)
# for name in sorted(type(settings).model_fields):
#     value = getattr(settings, name)
#     print(f"{name:<34}{value if value not in (None, '') else '(not set)'}")

In [16]:
from pydantic import ValidationError

from enterprise_agents_on_foundry.config.settings import Settings

# Frozen: a later stage of a run cannot quietly change what an earlier stage read.
try:
    settings.azure_location = "eastus"
except ValidationError as error:
    print(f"Mutation refused: {error.errors()[0]['type']}")

# Bounded: a row limit of zero is rejected at load, not discovered against the server.
try:
    Settings(database_max_result_rows=0)
except ValidationError as error:
    print(f"Invalid limit refused: {error.errors()[0]['msg']}")

Mutation refused: frozen_instance
Invalid limit refused: Input should be greater than or equal to 1


In [17]:
import tempfile

from enterprise_agents_on_foundry.errors import ConfigurationError

# A configuration error names the variable and the reason, and never the value.
with tempfile.TemporaryDirectory() as directory:
    bad_env = Path(directory) / ".env"
    bad_env.write_text("DATABASE_MAX_RESULT_ROWS=0\n", encoding="utf-8")
    try:
        load_settings(env_file=bad_env)
    except ConfigurationError as error:
        print(error)

## 5. Typed project errors

v0.1 raised a mixture of custom exceptions, bare `ValueError`, and `PermissionError`. That works until a caller
needs to distinguish *the environment is not ready* from *you asked for something unsafe*, because both arrive
as the same type from an unrelated part of the standard library.

Every error this project raises now inherits from `EaofError`, so a caller can catch everything the project
raises without also catching bugs. Five subclasses name the five distinct situations, and the choice of which
one to raise decides the exit code a command returns.

The `UnsafeDatabaseTargetError` case is the one worth dwelling on. Refusing to connect because the target is
wrong is not a failure of the check; it is the check working. It maps to exit code 2, *could not run*, rather
than exit code 1, *ran and failed*. Conflating those two is how a broken environment gets reported as a broken
database.

In [18]:
from enterprise_agents_on_foundry import errors

print(f"{errors.EaofError.__name__}")
for subclass in sorted(errors.EaofError.__subclasses__(), key=lambda item: item.__name__):
    summary = (subclass.__doc__ or "").strip().splitlines()[0]
    print(f"  {subclass.__name__:<28}{summary}")

EaofError
  AzureEnvironmentError       The Azure environment cannot be read or does not match configuration.
  ConfigurationError          Configuration is missing, malformed, or names an unsupported value.
  DatabaseConnectionError     A connection to Azure SQL could not be established or used.
  ModelOutputError            The model returned output that does not satisfy the expected schema.
  QueryValidationError        A statement is not a single, read-only query.
  UnsafeDatabaseTargetError   The configured database target is missing, malformed, or not permitted.


In [19]:
from enterprise_agents_on_foundry.database.validation import assert_read_only_sql, resolve_database_target
from enterprise_agents_on_foundry.errors import EaofError, QueryValidationError, UnsafeDatabaseTargetError

# Two different problems, two different types, one shared base to catch.
for attempt in [
    lambda: assert_read_only_sql("DELETE FROM SalesLT.Product"),
    lambda: resolve_database_target(Settings(azure_sql_server_fqdn="evil.example.com")),
]:
    try:
        attempt()
    except EaofError as error:
        print(f"{type(error).__name__}\n  {error}\n")

print(f"Both are EaofError    : {issubclass(QueryValidationError, EaofError)}")
print(f"Neither is a bare one : {not issubclass(UnsafeDatabaseTargetError, PermissionError)}")

QueryValidationError
  Query must begin with SELECT or WITH; found 'DELETE'.

UnsafeDatabaseTargetError
  AZURE_SQL_SERVER_FQDN must end with '.database.windows.net'; got 'evil.example.com'.

Both are EaofError    : True
Neither is a bare one : True


## 6. Loading azd deployment outputs

After `azd provision`, the deployment reports what it created. Two rules govern how that information is used.

Parsing is separated from invocation. `parse_azd_outputs` is a pure function over a string, so every rule about
malformed payloads, blank values, and non-object JSON is unit-tested without azd installed. `load_azd_outputs`
is the thin part that shells out.

Only keys already declared in `.env.example` are ever written to `.env`, and two keys are protected outright.
`ALLOW_DATABASE_BOOTSTRAP` is a local safety decision that a deployment has no business changing, and
`APPLICATIONINSIGHTS_CONNECTION_STRING` carries an instrumentation key.

Key names are printed below. Values are not, because deployment outputs are the kind of thing that gets pasted
into an issue.

In [20]:
from enterprise_agents_on_foundry.infrastructure.azd_outputs import PROTECTED_KEYS, load_azd_outputs

outputs = load_azd_outputs()

if outputs.is_empty:
    print("No azd environment found. Run 'azd provision' first.")
else:
    print(f"Deployment reported {len(outputs.names())} value(s). Names only:")
    for key in outputs.names():
        print(f"  {'protected' if key in PROTECTED_KEYS else 'writable ':<10} {key}")

Deployment reported 31 value(s). Names only:
  writable   AZURE_ENV_NAME
  writable   AZURE_FOUNDRY_ACCOUNT_ENDPOINT
  writable   AZURE_FOUNDRY_PROJECT_ENDPOINT
  writable   AZURE_FOUNDRY_PROJECT_NAME
  writable   AZURE_FOUNDRY_RESOURCE_NAME
  writable   AZURE_KEY_VAULT_NAME
  writable   AZURE_LOCATION
  writable   AZURE_LOG_ANALYTICS_WORKSPACE_NAME
  writable   AZURE_MANAGED_IDENTITY_CLIENT_ID
  writable   AZURE_MANAGED_IDENTITY_NAME
  writable   AZURE_MODEL_DEPLOYMENT_NAME
  writable   AZURE_MODEL_NAME
  writable   AZURE_MODEL_VERSION
  writable   AZURE_RESOURCE_GROUP
  writable   AZURE_RESOURCE_NAME_SUFFIX
  writable   AZURE_SQL_AUTHENTICATION
  writable   AZURE_SQL_DATABASE_NAME
  writable   AZURE_SQL_PUBLIC_NETWORK_ACCESS
  writable   AZURE_SQL_SERVER_FQDN
  writable   AZURE_SQL_SERVER_NAME
  writable   AZURE_SUBSCRIPTION_ID
  writable   AZURE_TENANT_ID
  writable   ENABLE_AZURE_AI_SEARCH
  writable   ENABLE_CONTAINER_REGISTRY
  writable   ENABLE_EXTERNAL_HOSTING
  writable   ENAB

In [14]:
import json

from enterprise_agents_on_foundry.errors import AzureEnvironmentError
from enterprise_agents_on_foundry.infrastructure.azd_outputs import parse_azd_outputs

# Parsing is pure, so its edge cases are demonstrable without a deployment.
print(f"Empty output is not an error : {parse_azd_outputs('').is_empty}")
print(f"Blank values read as absent  : {parse_azd_outputs(json.dumps({'AZURE_RESOURCE_GROUP': '  '})).resource_group}")

try:
    parse_azd_outputs("[1, 2, 3]")
except AzureEnvironmentError as error:
    print(f"Wrong JSON shape refused     : {error}")

Empty output is not an error : True
Blank values read as absent  : None
Wrong JSON shape refused     : azd returned list, but a JSON object was expected.


In [15]:
from enterprise_agents_on_foundry.infrastructure.environment import check_expected_resource_group

# The configured resource group is compared against the deployed one before anything reads or writes.
# Running validation against a different resource group than intended is how the wrong environment is changed.
check = check_expected_resource_group(settings.azure_resource_group, outputs.resource_group)
print(f"{'match' if check.matches else 'MISMATCH'}: {check.message}")
print()
print(check_expected_resource_group("rg-eaof-dev", "rg-eaof-prod").message)

match: Resource group 'rg-enterprise-agents-on-foundry-dev' matches the deployment outputs.

Configuration expects resource group 'rg-eaof-dev' but the azd environment deployed 'rg-eaof-prod'. Refusing to guess which one is correct.


## 7. Creating a database client

The database boundary is the only part of the package that opens a network connection, and it is deliberately
narrow. Three things happen before a socket is opened.

The target is **resolved and validated**. A hostname outside `database.windows.net`, a database name with
unexpected characters, or a server name that disagrees with the FQDN all stop here with a message, rather than
surfacing later as an ODBC error code.

The **driver is checked**. A missing ODBC driver produces `IM002` from pyodbc, which does not say which driver
was wanted or how to install it. The package checks first and explains.

The **credential is a token, not a password**. There is no password anywhere in this project: the Bicep
declares no password parameter, `Settings` has no field that could hold one, and a test asserts that. The
access token is attached through the ODBC attribute `SQL_COPT_SS_ACCESS_TOKEN`, which keeps it out of the
connection string and therefore out of anything that logs one.

In [21]:
from enterprise_agents_on_foundry.errors import DatabaseConnectionError

if settings.is_provisioned:
    target = resolve_database_target(settings)
    print(f"Target           : {target.display}")
    print(f"Connection string: {target.odbc_connection_string()}")
    print()
    print("No UID, no PWD. The token is attached out of band.")
else:
    print("Not provisioned. Missing:", ", ".join(settings.missing_provisioning_outputs()))

Target           : sql-eaof-dev-wgi4fh.database.windows.net/AdventureWorksLT (auth=entra)
Connection string: Driver={ODBC Driver 18 for SQL Server};Server=tcp:sql-eaof-dev-wgi4fh.database.windows.net,1433;Database=AdventureWorksLT;Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;

No UID, no PWD. The token is attached out of band.


In [24]:
from enterprise_agents_on_foundry.database.connection import connect

# The client is a context manager, so a connection cannot be left open by an exception
# or by a notebook cell that is never re-run.
client = None
try:
    client = connect(settings)
    print(f"Connected. Row limit for every query: {client.max_rows}")
except (DatabaseConnectionError, UnsafeDatabaseTargetError) as error:
    print(f"Cannot connect, and the reason is actionable:\n\n{error}")

Cannot connect, and the reason is actionable:

The 'ODBC Driver 18 for SQL Server' ODBC driver is not installed.
It is a system package, so 'uv sync' cannot provide it.

Windows: winget install --id Microsoft.msodbcsql18
macOS:   brew install msodbcsql18
Linux:   https://learn.microsoft.com/sql/connect/odbc/linux-mac/installing-the-microsoft-odbc-driver-for-sql-server

Installing it requires administrator rights. Open a new terminal afterwards so the driver registration is picked up.

Drivers currently visible: SQL Server, Microsoft Access Driver (*.mdb, *.accdb), Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb), Microsoft Access Text Driver (*.txt, *.csv), Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


## 8. Checking database health

`health_check()` answers one question and never raises: is the configured target reachable right now. It is
separate from querying because the answers to *is it reachable* and *did my query work* need different
responses, and because the Azure SQL database in this project is serverless.

Serverless auto-pauses after 60 idle minutes, and the first connection after a pause pays a resume cost of
several seconds. That is normal behaviour, not a fault, but it is indistinguishable from a hang if nothing
reports the elapsed time.

In [18]:
if client is not None:
    health = client.health_check()
    print(health.summary)
    if health.server_version:
        print(health.server_version)
    measurements.add("sql_health_check_latency", health.elapsed_ms, unit="ms", category="database")
else:
    print("Skipped: no connection.")

Skipped: no connection.


## 9. Listing AdventureWorksLT schemas and tables

Schema discovery is what a text-to-SQL agent needs before it can generate anything, so it belongs in the
package rather than in a prompt or a script. v0.3 will consume exactly these functions.

The SQL itself lives in `database/queries/*.sql`, not in Python string literals. One source of truth means the
same statements are reviewed as SQL, executed by the package, run by the command, and asserted read-only by a
test that walks the directory.

Row counts come from `sys.dm_db_partition_stats` rather than `COUNT(*)`. They are approximate, and they are
labelled as approximate, because scanning every table to produce an exact number on a serverless database that
bills per vCore-second is a poor trade for a fact nobody needs precisely.

In [19]:
from enterprise_agents_on_foundry.database.metadata import list_schemas, list_tables, total_approximate_rows

if client is not None:
    schemas = list_schemas(client)
    tables = list_tables(client)

    for schema in schemas:
        print(f"{schema.name:<24}{schema.table_count} table(s)")
    print()
    for table in tables:
        print(f"{table.qualified_name:<40}{table.approximate_rows:>8}")
    print()
    print(f"{len(tables)} tables, {total_approximate_rows(tables)} approximate rows.")

    measurements.add("schema_count", len(schemas), category="database")
    measurements.add("table_count", len(tables), category="database")
else:
    print("Skipped: no connection.")

Skipped: no connection.


## 10. Running safe read-only queries

Three independent controls apply to every query, and none of them depends on a prompt.

The **validator** rejects anything that is not a single `SELECT` or `WITH`. It strips comments before matching,
so a keyword cannot hide behind one, and it tracks string literals, so a semicolon inside a value is not read
as a statement separator.

The **row limit** is enforced by fetching one row more than the caller asked for. If that extra row exists, the
result is marked truncated. Silently returning fewer rows than exist is how a wrong answer looks correct.

The **database principal** holds `db_datareader` and is explicitly denied `INSERT`, `UPDATE`, `DELETE`,
`ALTER`, and `EXECUTE`, so a destructive statement fails at the server even if the first two controls were
somehow bypassed.

In [20]:
from enterprise_agents_on_foundry.database.models import QueryRequest

# Limits are validated when the request is built, before a connection is involved.
request = QueryRequest(sql="SELECT TOP (5) Name FROM SalesLT.Product", max_rows=5, label="sample products")
print(f"{request.label}: max_rows={request.max_rows}, timeout={request.timeout_seconds}s")

try:
    QueryRequest(sql="SELECT 1", max_rows=1_000_000)
except ValidationError as error:
    print(f"Unbounded request refused: {error.errors()[0]['msg']}")

sample products: max_rows=5, timeout=30s
Unbounded request refused: Input should be less than or equal to 10000


In [21]:
from enterprise_agents_on_foundry.database.metadata import run_smoke_query

if client is not None:
    result = run_smoke_query(client)
    print(result.format_table())
    print(f"\n{result.row_count} row(s) in {result.elapsed_ms} ms. Truncated: {result.truncated}")
    measurements.add("sql_simple_query_latency", result.elapsed_ms, unit="ms", category="database")
else:
    print("Skipped: no connection.")

Skipped: no connection.

In [22]:
if client is not None:
    # Truncation is reported rather than hidden.
    limited = client.execute(QueryRequest(sql="SELECT * FROM SalesLT.Product", max_rows=3))
    print(f"Returned {limited.row_count} row(s), truncated={limited.truncated}")

    # A write never reaches the server.
    try:
        client.execute_sql("UPDATE SalesLT.Product SET ListPrice = 0")
    except QueryValidationError as error:
        print(f"Refused before transmission: {error}")
else:
    print("Skipped: no connection.")

Skipped: no connection.


In [23]:
if client is not None:
    client.close()
    print("Connection closed. close() is idempotent, so re-running this cell is safe.")

## 11. Using the CLI entry points

Two commands are declared in `pyproject.toml` under `[project.scripts]`, so `uv sync` installs them into the
environment as real executables:

```text
uv run eaof-verify     # is this workstation and Azure environment ready
uv run eaof-db-info    # what is in the database, read-only
```

Both are thin: they own presentation and an exit code, and every fact they print comes from the package. That
is what stops the notebook, the pre-provision hook, and CI from disagreeing about what "ready" means.

The exit codes carry information, which matters the moment either command runs unattended:

| Code | Meaning | Correct response |
| --- | --- | --- |
| 0 | The check passed | Continue |
| 1 | The check ran and something failed | Fix the reported problem |
| 2 | The check could not run at all | Fix the environment, then re-run |

A tool that returns 1 for both "your database is broken" and "you have not installed a driver" sends people to
debug the wrong system.

In [24]:
from enterprise_agents_on_foundry.cli import EXIT_FAILED, EXIT_OK, EXIT_UNAVAILABLE, verify

exit_code = verify.main()

meaning = {EXIT_OK: "ready", EXIT_FAILED: "a check failed", EXIT_UNAVAILABLE: "could not run"}
print(f"\nExit code {exit_code}: {meaning[exit_code]}")


Configuration


-------------
  Project     : enterprise-agents-on-foundry
  Environment : dev
  Dataset     : adventureworks-lt
  Region      : westus3
  Resource grp: rg-enterprise-agents-on-foundry-dev
  Model       : gpt-5.4-mini on GlobalStandard



Workstation prerequisites
-------------------------
  python   ok        3.12.11
  git      ok        git version 2.55.0.windows.3
  uv       ok        uv 0.8.3 (7e78f54e7 2025-07-24)
  az       ok        azure-cli                         2.69.0 *
  azd      ok        azd version 1.28.1 (commit 3cf6db5881d8b24bb497e1470a972f4e28eb0256) (stable)
  bicep    ok        Bicep CLI version 0.45.15 (6a4a640fd8)

Azure sign-in
-------------
  Signed in to the expected subscription: MCAPS-Hybrid-ShivaChittamuru (1fad602f-d06f-46af-8f70-78a2c2c53b24) as shchitt@microsoft.com.

Deployment
----------
  Resource group 'rg-enterprise-agents-on-foundry-dev' matches the deployment outputs.

Result: READY
-------------
  Run 'uv run eaof-db-info' to inspect the database.

Exit code 0: ready


In [25]:
from enterprise_agents_on_foundry.cli import database_info

db_exit_code = database_info.main()
print(f"\nExit code {db_exit_code}: {meaning[db_exit_code]}")


Target
------
  sql-eaof-dev-wgi4fh.database.windows.net/AdventureWorksLT (auth=entra)
  Dataset: adventureworks-lt
  No password is used or stored. Access is Microsoft Entra only.

Result: UNAVAILABLE
-------------------
  The 'ODBC Driver 18 for SQL Server' ODBC driver is not installed.
It is a system package, so 'uv sync' cannot provide it.

Windows: winget install --id Microsoft.msodbcsql18
macOS:   brew install msodbcsql18
Linux:   https://learn.microsoft.com/sql/connect/odbc/linux-mac/installing-the-microsoft-odbc-driver-for-sql-server

Installing it requires administrator rights. Open a new terminal afterwards so the driver registration is picked up.

Drivers currently visible: SQL Server, Microsoft Access Driver (*.mdb, *.accdb), Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb), Microsoft Access Text Driver (*.txt, *.csv), Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)

Exit code 2: could not run


## 12. Running unit and integration tests

The suite is split by what it needs, not by what it covers.

```text
uv run pytest -m "not azure"   # tests/unit, no cloud access, seconds
uv run pytest -m azure         # tests/integration, needs a provisioned environment
uv run pytest                  # everything
```

The reason for the split is that a test suite which needs a subscription is a test suite that gets skipped. The
offline suite is the one that runs on every save and in every pull request, so anything that can be decided
without Azure is decided there: parsing, validation, comparison, limits, and exit codes are all pure functions
precisely so they can be.

The Azure suite asserts what only a live database can: that the connection works, that the schema is the one
expected, and that the row limit holds against the server rather than only in a unit test. It skips with a
reason rather than failing when the environment is absent, so a fresh clone can still run the whole command.

`--strict-markers` is set, so a typo in a marker name is an error rather than a silently unfiltered test.

In [ ]:
import shutil
import subprocess
import time

# Resolve the absolute path once rather than letting the operating system search
# PATH at call time. Ruff flags the partial-path form (S607) for exactly that
# reason: what runs then depends on the caller's environment.
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv was not found on PATH.")


def run_suite(marker: str) -> tuple[str, float]:
    started = time.perf_counter()
    completed = subprocess.run(  # noqa: S603
        [UV, "run", "pytest", "-m", marker, "-q", "--no-header"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    elapsed = (time.perf_counter() - started) * 1000
    summary = [line for line in completed.stdout.splitlines() if line.strip()][-1]
    return summary, elapsed


for marker, label in [("not azure", "unit"), ("azure", "integration")]:
    summary, elapsed = run_suite(marker)
    print(f"{label:<12}{summary}")
    measurements.add(f"{label}_test_duration", round(elapsed), unit="ms", category="quality")

unit        186 passed, 10 deselected in 7.04s


integration 1 passed, 9 skipped, 186 deselected in 0.48s


## 13. Before and after

The claim this release makes is that the code is easier to change safely. The measurements below are the
evidence for it, taken against v0.1 on the same machine.

Two numbers deserve comment.

Direct environment reads went from four to zero. Every one of those four was in a script, and each carried its
own inline default, so the model SKU was declared in one place and defaulted in another.

Duplicate helper implementations went from six to zero: one header printer defined twice, three near-identical
subprocess wrappers, and two model fallback constants. None of them were wrong. They were the second copy,
which is the one that stops matching the first.

In [27]:
BEFORE_AFTER = [
    ("direct_environment_reads", 4, 0, "count"),
    ("duplicate_helper_implementations", 6, 0, "count"),
    ("package_cli_commands", 0, 2, "count"),
    ("package_boundaries", 1, 5, "count"),
    ("ruff_findings", 0, 0, "count"),
    ("mypy_findings", 0, 0, "count"),
]

print(f"{'measure':<36}{'v0.1':>8}{'v0.2':>8}")
print("-" * 52)
for name, before, after, unit in BEFORE_AFTER:
    print(f"{name:<36}{before:>8}{after:>8}")
    measurements.add(name, after, unit=unit, category="refactor", baseline=before)

measure                                 v0.1    v0.2
----------------------------------------------------
direct_environment_reads                   4       0
duplicate_helper_implementations           6       0
package_cli_commands                       0       2
package_boundaries                         1       5
ruff_findings                              0       0
mypy_findings                              0       0


In [28]:
output_path = measurements.write_json(ROOT / "docs" / "releases" / "v0.2-measurements.json")
print(measurements.format_table())
print(f"\nWritten to {output_path.relative_to(ROOT)}")

category    measure                               before     after  unit
------------------------------------------------------------------------
quality     unit_test_duration                         -     10319  ms
quality     integration_test_duration                  -      5421  ms
refactor    direct_environment_reads                   4         0  count
refactor    duplicate_helper_implementations           6         0  count
refactor    package_cli_commands                       0         2  count
refactor    package_boundaries                         1         5  count
refactor    ruff_findings                              0         0  count
refactor    mypy_findings                              0         0  count

Written to docs\releases\v0.2-measurements.json


## What this release deliberately did not do

No LangGraph, no agent, no graph state, and no checkpointing. Those are v0.3, and the reason for the order is
that migrating an agent onto an untyped, untested foundation produces a migration that cannot be verified.

No FastAPI, no hosted agents, no Foundry IQ, no long-term memory, no MCP, no Teams distribution, and no
external hosting. Each has a release of its own on the roadmap.

No new Azure resources. The v0.1 foundation is unchanged, and every optional capability flag is still `false`.

One property is temporary and worth naming rather than leaving implicit: the Azure SQL server still has a
public endpoint restricted by firewall rule. That is appropriate for a single-developer teaching environment
and inappropriate for anything else. Private networking is parameterised in the Bicep already and defaults to
`false`; turning it on is a governance release, not a refactor.

Next: `docs/architecture/v0.2-python-foundation.md` for why each boundary exists, and
`docs/adr/005-use-src-layout-and-thin-adapters.md` for the decision record.